In [3]:
# This is a test because my model is underperforming
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import os

# from load import load_data, tensorize_data, clean_data
# from models import NaiveRNN, NaiveLSTM
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [2]:
con4_df = pd.read_csv(os.path.join("./Neuron Data", "PFC_con_4.csv"))
con4_df = con4_df.apply(pd.to_numeric, errors='coerce')

con4_df = clean_data(con4_df)

# Drop rat number, cell number, trial number
con4_df = con4_df.drop(columns=con4_df.columns[:3])
con4_df.columns = range(len(con4_df.columns))

# Split by trial type
con4_minus = con4_df[con4_df.iloc[:, 0] == 0]
con4_minus = con4_minus.drop(columns=con4_minus.columns[0])

# Extract labels
con4_minus_labels = con4_minus.iloc[:, 0].tolist()
con4_minus_labels = torch.tensor(con4_minus_labels, dtype=torch.long)

con4_minus = con4_minus.drop(columns=con4_minus.columns[0])

# Convert to tensors
con4_minus_tensor = torch.tensor(con4_minus.to_numpy(), dtype=torch.float32)

# Create datasets
con4_minus_dataset = TensorDataset(con4_minus_tensor, con4_minus_labels)

# Create dataloaders
con4_minus_loader = DataLoader(con4_minus_dataset, batch_size=32, shuffle=True)

In [3]:
print(con4_minus_tensor)
print(con4_minus_labels)

tensor([[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000, 50.3490],
        [-2.5175, -2.5175, -2.5175,  ..., -2.5175, -2.5175, -2.5175],
        [-5.0349, 20.1396, 20.1396,  ..., -5.0349, -5.0349, -5.0349],
        ...,
        [-1.2760, -1.2760, -1.2760,  ..., -1.2760, -1.2760, -1.2760],
        [-1.9140, -1.9140, -1.9140,  ..., -1.9140, 10.8458, -1.9140],
        [-3.1899, 22.3295, -3.1899,  ..., -3.1899,  9.5698, -3.1899]])
tensor([0, 1, 1,  ..., 1, 0, 0])


In [4]:
from sklearn.model_selection import train_test_split

# Trying to unsqueeze the tensor
con4_minus_tensor = con4_minus_tensor.unsqueeze(-1)

# Split dataset to training and validation
con4_minus_train, con4_minus_test, con4_minus_train_labels, con4_minus_test_labels = train_test_split(con4_minus_tensor, con4_minus_labels, test_size=0.2, random_state=42)

train_dataset = TensorDataset(con4_minus_train, con4_minus_train_labels)
test_dataset = TensorDataset(con4_minus_test, con4_minus_test_labels)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(con4_minus_labels)

tensor([0, 1, 1,  ..., 1, 0, 0])


In [5]:
"""Model Training"""
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Create models
rnn_model = NaiveRNN(1, 32, 2).to(device)
lstm_model = NaiveLSTM(1, 32, 2).to(device)
# cnn_model = NaiveCNN()

# Define loss and optimizer
criterion = nn.CrossEntropyLoss()
rnn_optimizer = optim.Adam(rnn_model.parameters(), lr=0.001)
lstm_optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)
# cnn_optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)

# Train the models
num_epochs = 20
for epoch in range(num_epochs):
    rnn_model.train()
    for inputs, labels in train_dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Convert to long tensor to avoid (problem with cross entropy)
        labels = labels.long()

        outputs = rnn_model(inputs)
        loss = criterion(outputs, labels)

        rnn_optimizer.zero_grad()
        loss.backward()
        rnn_optimizer.step()

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

    # Evaluation
rnn_model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for inputs, labels in test_dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        labels = labels.long()

        outputs = rnn_model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f'Accuracy: {100 * correct / total}%')

Epoch [1/20], Loss: 0.5022
Epoch [2/20], Loss: 0.2518
Epoch [3/20], Loss: 0.4948
Epoch [4/20], Loss: 0.5135
Epoch [5/20], Loss: 0.4639
Epoch [6/20], Loss: 0.4449
Epoch [7/20], Loss: 0.7103
Epoch [8/20], Loss: 0.3979
Epoch [9/20], Loss: 0.3890
Epoch [10/20], Loss: 0.3820
Epoch [11/20], Loss: 0.5083
Epoch [12/20], Loss: 0.3656
Epoch [13/20], Loss: 0.4776
Epoch [14/20], Loss: 0.5197
Epoch [15/20], Loss: 0.5377
Epoch [16/20], Loss: 0.4804
Epoch [17/20], Loss: 0.3837
Epoch [18/20], Loss: 0.4763
Epoch [19/20], Loss: 0.4372
Epoch [20/20], Loss: 0.5429
Accuracy: 77.9423226812159%
